# 01 — Quickstart

Install, env check, three operating modes (cloud/self-hosted/local), three voice modes (Realtime/ConvAI/Pipeline), 'hello phone' minimal agent.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

These cells require **T2** (local server) or **T3** (real API keys). Cells skip gracefully if prerequisites are missing.


### Agent object inspection


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI, DeepgramSTT, ElevenLabsTTS } from "getpatter";
await cell('agent_inspection', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'You are helpful.', engine: new OpenAIRealtime({ apiKey: 'sk-test' }), voice: 'alloy' });
  const pl = p.agent({ systemPrompt: 'Pipeline.', stt: new DeepgramSTT({ apiKey: 'dg-test' }), tts: new ElevenLabsTTS({ apiKey: 'el-test' }) });
  console.log(`realtime: provider=${rt.provider}  voice=${rt.voice}`);
  console.log(`pipeline: provider=${pl.provider}`);
});


### Pricing: calculate call costs


In [ ]:
import { DEFAULT_PRICING, calculateSttCost, calculateTtsCost, calculateTelephonyCost } from "getpatter";
await cell('pricing', { tier: 1, env }, () => {
  const stt = calculateSttCost('deepgram', 30, DEFAULT_PRICING);
  const tts = calculateTtsCost('elevenlabs', 200, DEFAULT_PRICING);
  const tel = calculateTelephonyCost('twilio', 60, DEFAULT_PRICING);
  console.log(`STT (Deepgram, 30s):         $${stt.toFixed(6)}`);
  console.log(`TTS (ElevenLabs, 200 chars): $${tts.toFixed(6)}`);
  console.log(`Telephony (Twilio, 60s):     $${tel.toFixed(6)}`);
});


### Text transforms


In [ ]:
import { filterMarkdown, filterEmoji, filterForTts } from "getpatter";
await cell('text_transforms', { tier: 1, env }, () => {
  const raw = '**Important**: Hello 😊 world';
  console.log(`filter_markdown: ${filterMarkdown(raw)}`);
  console.log(`filter_emoji:    ${filterEmoji(raw)}`);
  console.log(`filter_for_tts:  ${filterForTts(raw)}`);
});


### SentenceChunker


In [ ]:
import { SentenceChunker } from "getpatter";
await cell('sentence_chunker', { tier: 1, env }, () => {
  const sc = new SentenceChunker();
  const tokens = ['Hello', ' world', '!', ' How', ' are', ' you', '?'];
  const chunks: string[] = [];
  for (const tok of tokens) { chunks.push(...sc.push(tok)); }
  chunks.push(...sc.flush());
  console.log(`chunks: ${JSON.stringify(chunks)}`);
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.
